# Table Comparison: ds.silverdb vs prod_wis.silver_silverdb

This notebook compares tables between two catalogs:
- `ds.silverdb`
- `prod_wis.silver_silverdb`

Analysis includes:
1. Which tables exist in both/only one catalog
2. Schema comparison (field names)
3. Record counts
4. Unique participant_uuid counts
5. UUID overlap analysis

In [ ]:
from databricks import sql
import pandas as pd
import os
from typing import Set, Dict, List, Tuple
from dotenv import load_dotenv

load_dotenv()

# Connection setup
connection = sql.connect(
    server_hostname=os.getenv('DATABRICKS_SERVER_HOSTNAME'),
    http_path=os.getenv('DATABRICKS_HTTP_PATH'),
    access_token=os.getenv('DATABRICKS_TOKEN')
)

In [ ]:
def get_tables(catalog: str, schema: str) -> Set[str]:
    """Get list of table names from a catalog.schema"""
    with connection.cursor() as cursor:
        cursor.execute(f"SHOW TABLES IN {catalog}.{schema}")
        tables = {row.tableName for row in cursor.fetchall()}
    return tables

def get_schema(catalog: str, schema: str, table: str) -> Set[str]:
    """Get column names for a table"""
    with connection.cursor() as cursor:
        cursor.execute(f"DESCRIBE {catalog}.{schema}.{table}")
        columns = {row.col_name for row in cursor.fetchall() if not row.col_name.startswith('#')}
    return columns

def get_record_count(catalog: str, schema: str, table: str) -> int:
    """Get total record count"""
    with connection.cursor() as cursor:
        cursor.execute(f"SELECT COUNT(*) as cnt FROM {catalog}.{schema}.{table}")
        return cursor.fetchone().cnt

def get_unique_uuids(catalog: str, schema: str, table: str) -> int:
    """Get count of unique participant_uuid (if column exists)"""
    try:
        with connection.cursor() as cursor:
            cursor.execute(f"SELECT COUNT(DISTINCT participant_uuid) as cnt FROM {catalog}.{schema}.{table}")
            return cursor.fetchone().cnt
    except:
        return None

def get_uuid_overlap(catalog1: str, schema1: str, table1: str, 
                      catalog2: str, schema2: str, table2: str) -> int:
    """Get count of matching participant_uuid between two tables"""
    try:
        query = f"""
        SELECT COUNT(DISTINCT t1.participant_uuid) as overlap_count
        FROM {catalog1}.{schema1}.{table1} t1
        INNER JOIN {catalog2}.{schema2}.{table2} t2
        ON t1.participant_uuid = t2.participant_uuid
        """
        with connection.cursor() as cursor:
            cursor.execute(query)
            return cursor.fetchone().overlap_count
    except:
        return None

In [ ]:
# Get table lists
ds_tables = get_tables('ds', 'silverdb')
prod_tables = get_tables('prod_wis', 'silver_silverdb')

# Filter out quarantine tables from prod_wis for fairer comparison
prod_tables_clean = {t for t in prod_tables if not t.endswith('_quarantine')}

print(f"ds.silverdb tables: {len(ds_tables)}")
print(f"prod_wis.silver_silverdb tables (excl. quarantine): {len(prod_tables_clean)}")
print(f"prod_wis.silver_silverdb tables (incl. quarantine): {len(prod_tables)}")

In [ ]:
# Identify table overlaps
common_tables = ds_tables & prod_tables_clean
only_ds = ds_tables - prod_tables_clean
only_prod = prod_tables_clean - ds_tables

print(f"\nTables in both catalogs: {len(common_tables)}")
print(f"Tables only in ds.silverdb: {len(only_ds)}")
print(f"Tables only in prod_wis.silver_silverdb: {len(only_prod)}")

print(f"\n=== Only in ds.silverdb ===")
for t in sorted(only_ds):
    print(f"  - {t}")

print(f"\n=== Only in prod_wis.silver_silverdb ===")
for t in sorted(only_prod):
    print(f"  - {t}")

In [ ]:
# Build comprehensive comparison for common tables
comparison_data = []

for table in sorted(common_tables):
    print(f"Analyzing {table}...")
    
    # Get schemas
    ds_schema = get_schema('ds', 'silverdb', table)
    prod_schema = get_schema('prod_wis', 'silver_silverdb', table)
    
    schemas_match = ds_schema == prod_schema
    schema_diff = ""
    if not schemas_match:
        only_ds_cols = ds_schema - prod_schema
        only_prod_cols = prod_schema - ds_schema
        schema_diff = f"DS+: {only_ds_cols}, PROD+: {only_prod_cols}"
    
    # Get counts
    ds_count = get_record_count('ds', 'silverdb', table)
    prod_count = get_record_count('prod_wis', 'silver_silverdb', table)
    
    # Get unique UUIDs
    ds_uuids = get_unique_uuids('ds', 'silverdb', table)
    prod_uuids = get_unique_uuids('prod_wis', 'silver_silverdb', table)
    
    # Get UUID overlap
    uuid_overlap = None
    if ds_uuids is not None and prod_uuids is not None:
        uuid_overlap = get_uuid_overlap('ds', 'silverdb', table,
                                       'prod_wis', 'silver_silverdb', table)
    
    comparison_data.append({
        'table': table,
        'schemas_match': schemas_match,
        'schema_diff': schema_diff,
        'ds_records': ds_count,
        'prod_records': prod_count,
        'record_diff': ds_count - prod_count,
        'ds_unique_uuids': ds_uuids,
        'prod_unique_uuids': prod_uuids,
        'uuid_diff': (ds_uuids - prod_uuids) if (ds_uuids and prod_uuids) else None,
        'uuid_overlap': uuid_overlap
    })

df_comparison = pd.DataFrame(comparison_data)

In [ ]:
# Display full comparison table
print("\n" + "="*100)
print("COMPREHENSIVE TABLE COMPARISON")
print("="*100)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

display(df_comparison)

In [ ]:
# Summary statistics
print("\n" + "="*100)
print("SUMMARY STATISTICS")
print("="*100)

print(f"\nTables with matching schemas: {df_comparison['schemas_match'].sum()} / {len(df_comparison)}")
print(f"Tables with mismatched schemas: {(~df_comparison['schemas_match']).sum()}")

print(f"\nTables with more records in ds.silverdb: {(df_comparison['record_diff'] > 0).sum()}")
print(f"Tables with more records in prod_wis: {(df_comparison['record_diff'] < 0).sum()}")
print(f"Tables with same record count: {(df_comparison['record_diff'] == 0).sum()}")

has_uuid = df_comparison['ds_unique_uuids'].notna()
print(f"\nTables with participant_uuid column: {has_uuid.sum()} / {len(df_comparison)}")

if has_uuid.any():
    uuid_comparison = df_comparison[has_uuid]
    print(f"  - More UUIDs in ds.silverdb: {(uuid_comparison['uuid_diff'] > 0).sum()}")
    print(f"  - More UUIDs in prod_wis: {(uuid_comparison['uuid_diff'] < 0).sum()}")
    print(f"  - Same UUID count: {(uuid_comparison['uuid_diff'] == 0).sum()}")
    
    avg_overlap = uuid_comparison['uuid_overlap'].mean()
    print(f"\n  - Average UUID overlap: {avg_overlap:.0f} participants")

In [ ]:
# Export to CSV for further analysis
df_comparison.to_csv('table_comparison_summary.csv', index=False)
print("\nResults exported to: table_comparison_summary.csv")

In [ ]:
# Close connection
connection.close()
print("\nAnalysis complete!")